In [ ]:
velocity_nr = 127

filename = "velocity_reference_" + str(velocity_nr)
print(filename)

piano = "Shigeru_Kawai_SKEX"

def set_filenames():
    global filename
    global piano  
    global originalMIDI
    global originalTXT
    global originalPNG
    global originalAUDIO
    global generatedMIDI
    global generatedTXT
    global generatedPNG
    global generatedAUDIO
    global tempTXT
    global velocity_nr
    originalMIDI = filename + ".midi"
    originalTXT = filename + ".txt"
    originalPNG = filename + ".png"
    originalAUDIO = filename + ".wav"
    generatedAUDIO = filename + "_out.wav"
    generatedMIDI = filename + "_" + piano + ".mid"
    generatedPNG = filename + "_" + piano + ".png"
    generatedTXT = filename + "_" + piano + ".txt"
    tempTXT = "output.txt"
set_filenames()
print("original MIDI filename:", originalMIDI)
print("original TXT filename:", originalTXT)
print("original PNG filename:", originalPNG)
print("original AUDIO filename:", originalAUDIO)
print("generated MIDI filename:", generatedMIDI)
print("generated TXT filename:", generatedTXT)
print("generated PNG filename:", generatedPNG)
from IPython.display import Audio


In [ ]:
import os
import subprocess
from mido import Message, MidiFile, MidiTrack, MetaMessage


def generate_midi(constant_velocity):
    """
    Generates a MIDI file (and renders a WAV file using fluidsynth).

    Behavior:
    - tested notes are MIDI note numbers from 21 to 108
    - all notes use the same constant velocity
    - note 69 is inserted:
        * at the beginning
        * between consecutive tested notes
        * at the end
    - timing stays unchanged
    """

    # Hardcoded timing values (ticks)
    note_duration = 288  # on duration: 300ms
    gap_duration = 192   # off duration: 200ms



    # Create a new MIDI file and add two tracks
    mid = MidiFile(ticks_per_beat=480)
    piano_track = MidiTrack()
    pedal_track = MidiTrack()
    mid.tracks.append(piano_track)
    mid.tracks.append(pedal_track)

    # Set tempo: 500000 microseconds per beat = 120 BPM
    piano_track.append(MetaMessage('set_tempo', tempo=500000, time=0))

    # Tested notes: full piano range on 88-key piano
    tested_notes = list(range(21, 109))

    # Initial separator note 69 at the beginning
    piano_track.append(Message('note_on', note=69, velocity=constant_velocity, time=0))
    piano_track.append(Message('note_off', note=69, velocity=0, time=note_duration))

    # First tested note after a gap
    piano_track.append(Message('note_on', note=tested_notes[0], velocity=constant_velocity, time=gap_duration))
    piano_track.append(Message('note_off', note=tested_notes[0], velocity=0, time=note_duration))

    # For each next tested note, insert A4 (note 69) between consecutive tested notes
    for note_nr in tested_notes[1:]:
        piano_track.append(Message('note_on', note=69, velocity=constant_velocity, time=gap_duration))
        piano_track.append(Message('note_off', note=69, velocity=0, time=note_duration))

        piano_track.append(Message('note_on', note=note_nr, velocity=constant_velocity, time=gap_duration))
        piano_track.append(Message('note_off', note=note_nr, velocity=0, time=note_duration))

    # Final separator note 69 at the end
    piano_track.append(Message('note_on', note=69, velocity=constant_velocity, time=gap_duration))
    piano_track.append(Message('note_off', note=69, velocity=0, time=note_duration))

    # Save MIDI
    mid.save(originalMIDI)
    print(f"MIDI file '{originalMIDI}' generated successfully!")


    # Render the MIDI file to a WAV file using FluidSynth with the specified soundfont.
    fluidsynth_path = r"SOUNDFONTS/bin/fluidsynth.exe"
    soundfont_path = r"SOUNDFONTS/KAWAI_MP11SE_SK_Concert_Grand.sf2"

    fluidsynth_cmd = (
        f'"{fluidsynth_path}" -ni "{soundfont_path}" "{generatedMIDI}" '
        f'-F "{generatedAUDIO}" -r 44100 -g 1'
    )

    subprocess.run(fluidsynth_cmd, shell=True, check=True)
    print(f"WAV file '{generatedAUDIO}' generated successfully!")


# Call
generate_midi(constant_velocity=velocity_nr)

In [ ]:
def midi_to_text_and_plot(midi_path, text_path, plot_path, title_text):
    """
    Decodes a MIDI file, writes its events into a text file, and plots:
    1) Velocity vs. Time (ignoring velocity 0) with Y-axis from 0 to 127
    2) Histogram of Velocity Values with 128 bins and X-axis from 0 to 127
    3) MIDI Note Numbers vs. Time mapped to piano keys (1-88)
    4) Histogram of MIDI Note Numbers (mapped to 1-88) with 88 bins
    Args:
        midi_path (str): Path to the input MIDI file.
        text_path (str): Path to save the output text file.
        plot_path (str): Path to save the generated plot as a PNG file.
        title_text (str): Text to be displayed in the upper left corner of the plot.
    """
    import mido
    import matplotlib.pyplot as plt

    midi_file = mido.MidiFile(midi_path)
    ticks_per_beat = midi_file.ticks_per_beat
    tempo = 500000  # Default 120 BPM tempo (500,000 microseconds per beat)

    # Store extracted data
    velocity_values = []
    velocity_times = []
    note_numbers = []
    note_times = []

    with open(text_path, 'w') as text_file:
        text_file.write(f"MIDI File: {midi_path}\n")
        text_file.write(f"Ticks per Beat: {ticks_per_beat}\n\n")

        absolute_ticks = 0  # Tracks cumulative time in ticks

        for i, track in enumerate(midi_file.tracks):
            text_file.write(f"Track {i}: {track.name}\n")
            text_file.write("-" * 40 + "\n")

            for msg in track:
                absolute_ticks += msg.time  # Convert relative to absolute time

                # Tempo Change Handling (Optional)
                if msg.type == 'set_tempo':
                    tempo = msg.tempo  # Set new tempo

                # Convert MIDI ticks to seconds
                seconds = mido.tick2second(absolute_ticks, ticks_per_beat, tempo)

                # Note-On Event (Velocity > 0)
                if msg.type == 'note_on' and msg.velocity > 0:
                    velocity_values.append(msg.velocity)  # Store velocity
                    velocity_times.append(seconds)         # Store time for velocity
                    note_numbers.append(msg.note)            # Store MIDI note number
                    note_times.append(seconds)               # Store time for note

                    text_file.write(f"Time {seconds:.3f} sec | NOTE_ON  | "
                                    f"Note: {msg.note:3d} | Velocity: {msg.velocity:3d}\n")

                # Note-Off Event
                elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                    text_file.write(f"Time {seconds:.3f} sec | NOTE_OFF | "
                                    f"Note: {msg.note:3d} | Velocity: 0\n")

            text_file.write("\n")

    # Map MIDI note numbers to piano keys: MIDI 21 -> 1, MIDI 108 -> 88.
    mapped_note_numbers = [n - 20 for n in note_numbers]

    # Generate Plots as Subplots if there is any velocity data
    if velocity_values:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Top Left: Velocity vs. Time (set Y-axis from 0 to 127)
        axes[0, 0].scatter(velocity_times, velocity_values, color='b', alpha=0.7, label="Note Velocity")
        axes[0, 0].set_xlabel("Time (seconds)")
        axes[0, 0].set_ylabel("Velocity (0-127)")
        axes[0, 0].set_title("Velocity vs. Time")
        axes[0, 0].legend()
        axes[0, 0].grid()
        axes[0, 0].set_ylim(0, 127)
        axes[0, 0].set_xlim(left=0)
        axes[0, 0].text(-0.07, 1.1, title_text, transform=axes[0, 0].transAxes, fontsize=12, verticalalignment='top')

        # Bottom Left: MIDI Note Numbers vs. Time (mapped to 1-88)
        axes[1, 0].scatter(note_times, mapped_note_numbers, color='purple', alpha=0.7, label="Piano Key")
        axes[1, 0].set_xlabel("Time (seconds)")
        axes[1, 0].set_ylabel("Piano Key (1-88)")
        axes[1, 0].set_title("MIDI Note Numbers vs. Time")
        axes[1, 0].legend()
        axes[1, 0].grid()
        axes[1, 0].set_xlim(left=0)
        axes[1, 0].set_ylim(1, 88)

        # Top Right: Histogram of Velocity Values (set X-axis from 0 to 127)
        axes[0, 1].hist(velocity_values, bins=128, range=(0, 127), color='g', alpha=0.7, edgecolor='black')
        axes[0, 1].set_xlabel("Velocity")
        axes[0, 1].set_ylabel("Number of Occurrences")
        axes[0, 1].set_title("Histogram of Velocity Values")
        axes[0, 1].grid()
        axes[0, 1].set_xlim(0, 127)

        # Bottom Right: Histogram of MIDI Note Numbers (mapped to 1-88)
        axes[1, 1].hist(mapped_note_numbers, bins=88, range=(1, 88), color='orange', alpha=0.7, edgecolor='black')
        axes[1, 1].set_xlabel("Piano Key (1-88)")
        axes[1, 1].set_ylabel("Number of Occurrences")
        axes[1, 1].set_title("Histogram of MIDI Note Numbers")
        axes[1, 1].grid()
        axes[1, 1].set_xlim(1, 88)

        plt.tight_layout()
        plt.savefig(plot_path)  # Save plot as PNG file
        plt.show()

    print(f"PLOTS saved to: {plot_path}")
    print(f"MIDI events saved to: {text_path}")
midi_to_text_and_plot(originalMIDI, originalTXT, originalPNG, filename )

In [ ]:
print(originalAUDIO)
Audio(originalAUDIO)

In [ ]:
import sys
!{sys.executable}  example.py --audio_path={originalAUDIO} --output_midi_path={generatedMIDI}

In [ ]:
def midi_to_text_and_plot(midi_path, text_path, plot_path, title_text):
    """
    Decodes a MIDI file, writes its events into a text file, and plots:
    1) Velocity vs. Time (ignoring velocity 0) with Y-axis from 0 to 127
    2) Histogram of Velocity Values with 128 bins and X-axis from 0 to 127
    3) MIDI Note Numbers vs. Time mapped to piano keys (1-88)
    4) Histogram of MIDI Note Numbers (mapped to 1-88) with 88 bins
    Args:
        midi_path (str): Path to the input MIDI file.
        text_path (str): Path to save the output text file.
        plot_path (str): Path to save the generated plot as a PNG file.
        title_text (str): Text to be displayed in the upper left corner of the plot.
    """
    import mido
    import matplotlib.pyplot as plt

    midi_file = mido.MidiFile(midi_path)
    ticks_per_beat = midi_file.ticks_per_beat
    tempo = 500000  # Default 120 BPM tempo (500,000 microseconds per beat)

    # Store extracted data
    velocity_values = []
    velocity_times = []
    note_numbers = []
    note_times = []

    with open(text_path, 'w') as text_file:
        text_file.write(f"MIDI File: {midi_path}\n")
        text_file.write(f"Ticks per Beat: {ticks_per_beat}\n\n")

        absolute_ticks = 0  # Tracks cumulative time in ticks

        for i, track in enumerate(midi_file.tracks):
            text_file.write(f"Track {i}: {track.name}\n")
            text_file.write("-" * 40 + "\n")

            for msg in track:
                absolute_ticks += msg.time  # Convert relative to absolute time

                # Tempo Change Handling (Optional)
                if msg.type == 'set_tempo':
                    tempo = msg.tempo  # Set new tempo

                # Convert MIDI ticks to seconds
                seconds = mido.tick2second(absolute_ticks, ticks_per_beat, tempo)

                # Note-On Event (Velocity > 0)
                if msg.type == 'note_on' and msg.velocity > 0:
                    velocity_values.append(msg.velocity)  # Store velocity
                    velocity_times.append(seconds)         # Store time for velocity
                    note_numbers.append(msg.note)            # Store MIDI note number
                    note_times.append(seconds)               # Store time for note

                    text_file.write(f"Time {seconds:.3f} sec | NOTE_ON  | "
                                    f"Note: {msg.note:3d} | Velocity: {msg.velocity:3d}\n")

                # Note-Off Event
                elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                    text_file.write(f"Time {seconds:.3f} sec | NOTE_OFF | "
                                    f"Note: {msg.note:3d} | Velocity: 0\n")

            text_file.write("\n")

    # Map MIDI note numbers to piano keys: MIDI 21 -> 1, MIDI 108 -> 88.
    mapped_note_numbers = [n - 20 for n in note_numbers]

    # Generate Plots as Subplots if there is any velocity data
    if velocity_values:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Top Left: Velocity vs. Time (set Y-axis from 0 to 127)
        axes[0, 0].scatter(velocity_times, velocity_values, color='b', alpha=0.7, label="Note Velocity")
        axes[0, 0].set_xlabel("Time (seconds)")
        axes[0, 0].set_ylabel("Velocity (0-127)")
        axes[0, 0].set_title("Velocity vs. Time")
        axes[0, 0].legend()
        axes[0, 0].grid()
        axes[0, 0].set_ylim(0, 127)
        axes[0, 0].set_xlim(left=0)
        axes[0, 0].text(-0.07, 1.1, title_text, transform=axes[0, 0].transAxes, fontsize=12, verticalalignment='top')

        # Bottom Left: MIDI Note Numbers vs. Time (mapped to 1-88)
        axes[1, 0].scatter(note_times, mapped_note_numbers, color='purple', alpha=0.7, label="Piano Key")
        axes[1, 0].set_xlabel("Time (seconds)")
        axes[1, 0].set_ylabel("Piano Key (1-88)")
        axes[1, 0].set_title("MIDI Note Numbers vs. Time")
        axes[1, 0].legend()
        axes[1, 0].grid()
        axes[1, 0].set_xlim(left=0)
        axes[1, 0].set_ylim(1, 88)

        # Top Right: Histogram of Velocity Values (set X-axis from 0 to 127)
        axes[0, 1].hist(velocity_values, bins=128, range=(0, 127), color='g', alpha=0.7, edgecolor='black')
        axes[0, 1].set_xlabel("Velocity")
        axes[0, 1].set_ylabel("Number of Occurrences")
        axes[0, 1].set_title("Histogram of Velocity Values")
        axes[0, 1].grid()
        axes[0, 1].set_xlim(0, 127)

        # Bottom Right: Histogram of MIDI Note Numbers (mapped to 1-88)
        axes[1, 1].hist(mapped_note_numbers, bins=88, range=(1, 88), color='orange', alpha=0.7, edgecolor='black')
        axes[1, 1].set_xlabel("Piano Key (1-88)")
        axes[1, 1].set_ylabel("Number of Occurrences")
        axes[1, 1].set_title("Histogram of MIDI Note Numbers")
        axes[1, 1].grid()
        axes[1, 1].set_xlim(1, 88)

        plt.tight_layout()
        plt.savefig(plot_path)  # Save plot as PNG file
        plt.show()

    print(f"PLOTS saved to: {plot_path}")
    print(f"MIDI events saved to: {text_path}")
midi_to_text_and_plot(generatedMIDI, generatedTXT, generatedPNG, filename )

In [ ]:
print(generatedTXT)